In [ ]:
"""
K_A versus PARTIAL (i-j) lipid-lipid interaction energy.

    (a) all 20 systems on one set of axes
    (b) 65, 72, 80, 87 and 95 mol % DOPC shown separately, each with a
        least-squares line and R^2

    python fig_ka_vs_partial.py   ->  KA_vs_Partial.{png,pdf}
                                      KA_vs_Partial_fits.txt

Self-contained: the energies and K_A values are embedded below, so this runs
with nothing but numpy and matplotlib.

PARTIAL means the sum of the 24 UNLIKE subunit pairs, per Langmuir 2025, 41,
2600-2618: four subunits per lipid (head / phosphate / glycerol / acyl tail),
with head-head, phos-phos, glyc-glyc and tail-tail excluded.  6 pairs within
DOPC + 6 within the secondary lipid + 12 across.  The quantity plotted is the
overall lipid-lipid total, i.e. panel (d) of the decomposed figure.

=============================================================================
READ THIS BEFORE QUOTING AN R^2.

Chain length barely moves the partial energy -- DOPG/DLPG and DOPA/DLPA agree
to better than 0.5% on this quantity at every composition.  So although there
are four lipids per composition, they collapse onto TWO distinct x positions,
one PG and one PA.  Fitting a straight line through two clusters is not a
correlation in the usual sense: the slope is fixed by the two cluster means
and R^2 is then governed by how much the two K_A values within each cluster
happen to disagree.

At 65% the two clusters differ by about 9 mN/m in mean K_A while the spread
WITHIN each cluster is about 21 mN/m, so the fit is dominated by scatter and
R^2 collapses.  This is a property of the lipid set, not of the analysis.

The published study spanned five different headgroups, giving five distinct x
positions, which is why its R^2 values are high and ours are not comparable.

The script therefore prints a diagnostic alongside each R^2: the between-
cluster K_A separation, the within-cluster spread, and their ratio.  Where
that ratio is below ~1 the R^2 should not be presented as evidence of a
correlation.  Consider plotting the two families as groups instead.
=============================================================================
"""
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ systems
COMPS  = [65, 72, 80, 87, 95]                 # mol % DOPC
LIPIDS = ["DOPG", "DOPA", "DLPG", "DLPA"]
FAMILY = {"DOPG": "PG", "DLPG": "PG", "DOPA": "PA", "DLPA": "PA"}

# CAPTION: compositions are 65/72/80/87/95 mol % DOPC.  The published series
# was 65/75/85/95 -- do not copy that caption.

COLORS  = {"DOPG": "#E8000B", "DOPA": "#1F4E79",
           "DLPG": "#4C9A2A", "DLPA": "#E0A500"}
MARKERS = {"DOPG": "o", "DOPA": "s", "DLPG": "^", "DLPA": "D"}
FITCOL  = "#8B4513"

K = 4184.0            # kJ/mol -> x10^3 kcal/mol

# --------------------------------------------- partial (i-j) energy, kJ/mol
# Overall lipid-lipid, 24 unlike subunit pairs.  Order: 65, 72, 80, 87, 95.
PARTIAL_KJ = {
    "DOPG": [-170638.3, -168648.7, -166374.3, -164061.2, -161316.6],
    "DOPA": [-126348.5, -133209.4, -140869.7, -147365.2, -155282.1],
    "DLPG": [-171291.3, -169049.6, -166605.5, -164223.3, -161330.1],
    "DLPA": [-126963.9, -133748.4, -141058.9, -147564.7, -155312.2],
}

# ------------------------------------------------------------------ K_A
KA = {"DOPG": [236.7, 223.3, 217.3, 217.0, 260.0],
      "DOPA": [246.4, 250.5, 228.9, 261.8, 247.1],
      "DLPG": [216.2, 209.3, 274.1, 246.0, 287.3],
      "DLPA": [224.9, 269.7, 221.0, 264.8, 262.0]}
KA_ERR = {"DOPG": [35.9, 12.1, 14.9, 13.5, 20.5],
          "DOPA": [13.0, 38.5, 14.4, 12.1, 17.2],
          "DLPG": [23.6, 14.4, 16.5, 12.4, 30.3],
          "DLPA": [20.7, 12.8, 16.7, 16.0, 14.7]}
KA_PURE_DOPC = (245.81, 9.85)

# Pure DOPC partial energy, x10^3 kcal/mol.  This is an EXTENSIVE quantity and
# cannot be read off the published figure -- that bilayer is not necessarily
# our system size.  Fill from Table S3 of the Supporting Information.  While
# None, the black reference point is omitted.
PURE_DOPC_PARTIAL = None

XLABEL = r"Partial ($i$–$j$) interaction energy ($\times$10$^3$ kcal/mol)"
YLABEL = r"$K_A$ (mN/m)"

E = {l: np.array(PARTIAL_KJ[l], float) / K for l in LIPIDS}


# ------------------------------------------------------------------ helpers
def linfit(x, y):
    """Least-squares line + R^2.  None if fewer than 3 usable points."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3 or x.std() == 0:
        return None
    s, i = np.polyfit(x, y, 1)
    return s, i, np.corrcoef(x, y)[0, 1] ** 2


def cluster_diagnostic(j):
    """
    Between-cluster K_A separation vs within-cluster spread at composition j.
    Ratio < 1 means the scatter inside a family exceeds the difference
    between families, so the fit is not measuring a trend.
    """
    fam = {}
    for l in LIPIDS:
        fam.setdefault(FAMILY[l], []).append((E[l][j], KA[l][j]))
    if len(fam) < 2:
        return None
    means = {f: np.mean([k for _, k in v]) for f, v in fam.items()}
    spreads = [abs(v[0][1] - v[1][1]) for v in fam.values() if len(v) == 2]
    sep = abs(means["PA"] - means["PG"])
    spread = float(np.mean(spreads)) if spreads else np.nan
    xsep = abs(np.mean([x for x, _ in fam["PA"]]) -
               np.mean([x for x, _ in fam["PG"]]))
    return sep, spread, (sep / spread if spread else np.nan), xsep


# ------------------------------------------------------------------ figure
def main():
    # 2x5 grid: (a) spans the left two columns, the five compositions take the
    # remaining cells, and the spare cell carries the footnote.
    fig = plt.figure(figsize=(15.5, 6.4))
    gs = fig.add_gridspec(2, 5, width_ratios=[1.25, 1.25, 1, 1, 1],
                          hspace=0.46, wspace=0.36,
                          left=0.055, right=0.985, bottom=0.10, top=0.90)
    ax = fig.add_subplot(gs[:, 0:2])

    # ---- (a) every system -------------------------------------------------
    for l in LIPIDS:
        ax.errorbar(E[l], KA[l], yerr=KA_ERR[l], fmt=MARKERS[l], ms=8,
                    color=COLORS[l], markeredgecolor="k", markeredgewidth=0.6,
                    ecolor="0.65", elinewidth=0.9, capsize=2, lw=0, label=l)
    if PURE_DOPC_PARTIAL is not None:
        ax.errorbar([PURE_DOPC_PARTIAL], [KA_PURE_DOPC[0]],
                    yerr=[KA_PURE_DOPC[1]], fmt="o", ms=9, color="k",
                    ecolor="0.5", capsize=2, label="Pure DOPC")
    ax.set_xlabel(XLABEL, fontsize=10)
    ax.set_ylabel(YLABEL)
    ax.text(0.03, 0.96, "(a)", transform=ax.transAxes, fontsize=13,
            fontweight="bold", va="top")
    ax.legend(fontsize=8.5, ncol=2)
    ax.tick_params(direction="in", top=True, right=True)

    # ---- (b) one panel per composition ------------------------------------
    CELLS = [(0, 2), (0, 3), (0, 4), (1, 2), (1, 3)]   # (1,4) is the note cell
    XLAB  = {2, 3, 4}          # panels with nothing beneath them in the grid
    lines = []
    for k, comp in enumerate(COMPS):
        r, c = CELLS[k]
        a2 = fig.add_subplot(gs[r, c])
        xs, ys = [], []
        for l in LIPIDS:
            xs.append(E[l][k]); ys.append(KA[l][k])
            a2.plot(E[l][k], KA[l][k], MARKERS[l], ms=7, color=COLORS[l],
                    markeredgecolor="k", markeredgewidth=0.5)
        if PURE_DOPC_PARTIAL is not None:
            xs.append(PURE_DOPC_PARTIAL); ys.append(KA_PURE_DOPC[0])
            a2.plot(PURE_DOPC_PARTIAL, KA_PURE_DOPC[0], "o", ms=7, color="k")

        f = linfit(xs, ys)
        diag = cluster_diagnostic(k)
        if f:
            s, i, R2 = f
            xf = np.linspace(min(xs), max(xs), 50)
            a2.plot(xf, s * xf + i, ls="--", lw=1.3, color=FITCOL)
            a2.text(0.5, 0.05, f"R$^2$ = {R2:.3f}",
                    transform=a2.transAxes, ha="center", fontsize=9,
                    bbox=dict(facecolor="white", edgecolor="none",
                              alpha=0.85, pad=1.5))
            lines.append((comp, R2, diag))
        a2.text(0.95, 0.92, f"{comp}%", transform=a2.transAxes, ha="right",
                va="top", fontsize=11, fontweight="bold")
        a2.tick_params(direction="in", top=True, right=True, labelsize=9)
        a2.ticklabel_format(useOffset=False)
        if c == 2:
            a2.set_ylabel(YLABEL, fontsize=10)
        if k in XLAB:
            # short form: the full wording is on panel (a) and would clip in
            # the rightmost column
            a2.set_xlabel(r"Partial energy ($\times$10$^3$ kcal/mol)",
                          fontsize=8.5)
        if k == 0:
            a2.text(-0.30, 1.22, "(b)", transform=a2.transAxes, fontsize=13,
                    fontweight="bold", va="top", ha="left")
    # The (1,4) cell is left empty on purpose.  The caveat about the cluster
    # structure lives in KA_vs_Partial_fits.txt, not on the figure.

    plt.savefig("KA_vs_Partial.png", dpi=600, bbox_inches="tight")
    plt.savefig("KA_vs_Partial.pdf", bbox_inches="tight")
    plt.close(fig)
    print("wrote KA_vs_Partial.png / .pdf")

    # ---- fit report -------------------------------------------------------
    with open("KA_vs_Partial_fits.txt", "w") as fh:
        fh.write("K_A vs PARTIAL (i-j) lipid-lipid interaction energy\n")
        fh.write("Least-squares fit per composition, 4 systems each.\n\n")
        fh.write(f"{'DOPC%':>6}{'R^2':>8}{'dKA(PA-PG)':>13}"
                 f"{'within-fam':>12}{'ratio':>8}{'dx':>8}\n")
        fh.write("-" * 55 + "\n")
        for comp, R2, d in lines:
            if d is None:
                fh.write(f"{comp:>5}%{R2:>8.3f}\n")
                continue
            sep, spread, ratio, xsep = d
            fh.write(f"{comp:>5}%{R2:>8.3f}{sep:>13.1f}{spread:>12.1f}"
                     f"{ratio:>8.2f}{xsep:>8.2f}\n")
        fh.write(
            "\ndKA(PA-PG)  difference in mean K_A between the PA and PG "
            "families (mN/m)\n"
            "within-fam  mean |K_A| spread between the two chain lengths "
            "inside a family\n"
            "ratio       dKA / within-fam.  Below 1 means the scatter inside "
            "a family exceeds\n"
            "            the difference between families, so the line is "
            "fitting noise.\n"
            "dx          separation of the two x clusters "
            "(x10^3 kcal/mol)\n\n"
            "Chain length does not move the partial energy (<0.5% at every "
            "composition), so\n"
            "the four lipids sit at only TWO x positions.  A line through two "
            "clusters has its\n"
            "slope set by the cluster means; R^2 then reports within-cluster "
            "K_A scatter, not\n"
            "the strength of any energy-K_A relationship.  Where the ratio "
            "is below 1, present\n"
            "the two families as groups rather than quoting the fit.\n")
    print("wrote KA_vs_Partial_fits.txt")
    for comp, R2, d in lines:
        tag = "" if (d is None or d[2] >= 1.0) else "   <-- fit not supported"
        print(f"  {comp}%  R^2 = {R2:.3f}" +
              ("" if d is None else f"   dKA={d[0]:.1f}  spread={d[1]:.1f}  "
                                    f"ratio={d[2]:.2f}") + tag)


if __name__ == "__main__":
    main()

In [ ]:
"""
K_A versus TOTAL lipid-lipid interaction energy.

    (a) all 20 systems on one set of axes
    (b) 65, 72, 80, 87 and 95 mol % DOPC shown separately, each with a
        least-squares line and R^2

    python fig_ka_vs_total.py   ->  KA_vs_Total.{png,pdf}
                                    KA_vs_Total_fits.txt

Self-contained: the energies and K_A values are embedded below, so this runs
with nothing but numpy and matplotlib.

TOTAL means every atom pair between the two lipids, from the whole-lipid
reruns (energygrps = DOPC <sec> IONS).  Contrast fig_ka_vs_partial.py, which
sums only the 24 unlike subunit pairs.

=============================================================================
TWO REASONS TO PREFER THE PARTIAL VERSION OF THIS FIGURE.

1. The totals include the diagonal (i-i) subunit terms, which carry the PME
   Ewald self-energy, -(1/4pi eps0)(beta/sqrt pi) * sum_i q_i^2.  That is an
   artifact of the group decomposition, not an interaction.  It is large: for
   100 DOPG lipids sum q_i^2 is roughly 650 e^2, worth about -132,000 kJ/mol.
   The partial definition excludes all i-i pairs and is free of it.

2. Because the self term scales with ATOM COUNT, it makes the totals depend on
   chain length for a non-physical reason.  DOPG has 131 atoms per lipid and
   DLPG 99, so DOPG carries a bigger self-energy and its total comes out more
   negative -- a 3-6% chain-length "effect" that is pure bookkeeping.  The
   partial energies show <0.5% chain-length dependence on the same systems.

   So this figure appears to have four distinct x positions per composition
   where the partial figure has two.  That extra spread is not signal.  Do not
   read it as chain-length discrimination.

Also note Coul-SR is only the PME real-space part throughout; the reciprocal
term is not decomposable by energy group and is absent from both figures.
=============================================================================
"""
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ systems
COMPS  = [65, 72, 80, 87, 95]                 # mol % DOPC
LIPIDS = ["DOPG", "DOPA", "DLPG", "DLPA"]
FAMILY = {"DOPG": "PG", "DLPG": "PG", "DOPA": "PA", "DLPA": "PA"}

# CAPTION: compositions are 65/72/80/87/95 mol % DOPC.  The published series
# was 65/75/85/95 -- do not copy that caption.

COLORS  = {"DOPG": "#E8000B", "DOPA": "#1F4E79",
           "DLPG": "#4C9A2A", "DLPA": "#E0A500"}
MARKERS = {"DOPG": "o", "DOPA": "s", "DLPG": "^", "DLPA": "D"}
FITCOL  = "#8B4513"

K = 4184.0            # kJ/mol -> x10^3 kcal/mol

# --------------------------------------------------- total energy, kJ/mol
# Whole-lipid rerun averages, in the order
#   Coul:PC-PC, LJ:PC-PC, Coul:PC-sec, LJ:PC-sec, Coul:sec-sec, LJ:sec-sec
# Summed at load time rather than pre-added, so the decomposition stays
# visible and the arithmetic is done once, in code.
#
# DOPA 80/20 and 87/13 were run on the SAME 201 frames as their subunit
# reruns; the subunit sums reproduce them to 0.0003% or better.
RAW_TOTAL = {
 "DOPG": {65: [45019.9, -28410,   -4958.89, -22421.4, -100827,  -9759.63],
          72: [49194.1, -34103.1, -4358.45, -19494.7, -80593.1, -7038.99],
          80: [53637.6, -40710.9, -3577.05, -15651.1, -58391.5, -4400.82],
          87: [57586.3, -47184.7, -2544.44, -11061.1, -38230.9, -2468.97],
          95: [62082.6, -55573.1, -1053.73, -4414.56, -14073.7, -743.869]},
 "DOPA": {65: [45226.9, -28471,   -4489.71, -22208,   -60499.8, -8959.79],
          72: [49283.5, -34290.5, -3936.44, -19034.3, -48454.4, -6550.99],
          80: [53635.5, -40931.7, -3223.57, -15144.8, -35173.1, -4091.37],
          87: [57664.6, -47286.6, -2338.58, -10854.6, -23044.3, -2228.09],
          95: [62164.4, -55607.9, -949.57,  -4345.19, -8506.74, -645.619]},
 "DLPG": {65: [44959,   -30674.9, -5172.21, -17496.1, -101736,  -6768.17],
          72: [49097.5, -36124.1, -4614.56, -15118.3, -81281.2, -4913.37],
          80: [53636,   -42293.5, -3829.08, -12126.1, -58921.2, -3016.11],
          87: [57518.3, -48497.5, -2701.89, -8316.2,  -38551.8, -1813.68],
          95: [62215.9, -56049.5, -1111.97, -3362.99, -14183.5, -525.855]},
 "DLPA": {65: [45022.4, -30890.8, -4728.19, -16994.1, -61348.6, -6198.91],
          72: [49244,   -36166.4, -4207,    -15000,   -49132.1, -4261.39],
          80: [53651.2, -42638,   -3380.03, -11417.7, -35641.6, -2824.09],
          87: [57566.5, -48648.4, -2436.71, -7977.37, -23372.4, -1617.07],
          95: [62132.6, -56151.8, -1008.6,  -3170.62, -8613.16, -487.34]},
}

# ------------------------------------------------------------------ K_A
KA = {"DOPG": [236.7, 223.3, 217.3, 217.0, 260.0],
      "DOPA": [246.4, 250.5, 228.9, 261.8, 247.1],
      "DLPG": [216.2, 209.3, 274.1, 246.0, 287.3],
      "DLPA": [224.9, 269.7, 221.0, 264.8, 262.0]}
KA_ERR = {"DOPG": [35.9, 12.1, 14.9, 13.5, 20.5],
          "DOPA": [13.0, 38.5, 14.4, 12.1, 17.2],
          "DLPG": [23.6, 14.4, 16.5, 12.4, 30.3],
          "DLPA": [20.7, 12.8, 16.7, 16.0, 14.7]}
KA_PURE_DOPC = (245.81, 9.85)

# Pure DOPC total energy, x10^3 kcal/mol.  EXTENSIVE quantity -- it cannot be
# read off the published figure, whose bilayer is not necessarily our system
# size.  Fill from Table S3 of the Supporting Information.  While None, the
# black reference point is omitted.
PURE_DOPC_TOTAL = None

XLABEL  = r"Total interaction energy ($\times$10$^3$ kcal/mol)"
XSHORT  = r"Total energy ($\times$10$^3$ kcal/mol)"
YLABEL  = r"$K_A$ (mN/m)"

E = {l: np.array([sum(RAW_TOTAL[l][c]) for c in COMPS], float) / K
     for l in LIPIDS}


# ------------------------------------------------------------------ helpers
def linfit(x, y):
    """Least-squares line + R^2.  None if fewer than 3 usable points."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3 or x.std() == 0:
        return None
    s, i = np.polyfit(x, y, 1)
    return s, i, np.corrcoef(x, y)[0, 1] ** 2


def cluster_diagnostic(j):
    """
    Between-family K_A separation vs within-family spread at composition j.
    Ratio < 1 means the scatter inside a family exceeds the difference
    between families, so the line is fitting noise rather than a trend.
    """
    fam = {}
    for l in LIPIDS:
        fam.setdefault(FAMILY[l], []).append((E[l][j], KA[l][j]))
    if len(fam) < 2:
        return None
    means = {f: np.mean([k for _, k in v]) for f, v in fam.items()}
    spreads = [abs(v[0][1] - v[1][1]) for v in fam.values() if len(v) == 2]
    sep = abs(means["PA"] - means["PG"])
    spread = float(np.mean(spreads)) if spreads else np.nan
    xsep = abs(np.mean([x for x, _ in fam["PA"]]) -
               np.mean([x for x, _ in fam["PG"]]))
    xin = float(np.mean([abs(v[0][0] - v[1][0])
                         for v in fam.values() if len(v) == 2]))
    return sep, spread, (sep / spread if spread else np.nan), xsep, xin


# ------------------------------------------------------------------ figure
def main():
    # 2x5 grid: (a) spans the left two columns, the five compositions take the
    # remaining cells.  The (1,4) cell is deliberately left empty.
    fig = plt.figure(figsize=(15.5, 6.4))
    gs = fig.add_gridspec(2, 5, width_ratios=[1.25, 1.25, 1, 1, 1],
                          hspace=0.46, wspace=0.36,
                          left=0.055, right=0.985, bottom=0.10, top=0.90)
    ax = fig.add_subplot(gs[:, 0:2])

    # ---- (a) every system -------------------------------------------------
    for l in LIPIDS:
        ax.errorbar(E[l], KA[l], yerr=KA_ERR[l], fmt=MARKERS[l], ms=8,
                    color=COLORS[l], markeredgecolor="k", markeredgewidth=0.6,
                    ecolor="0.65", elinewidth=0.9, capsize=2, lw=0, label=l)
    if PURE_DOPC_TOTAL is not None:
        ax.errorbar([PURE_DOPC_TOTAL], [KA_PURE_DOPC[0]],
                    yerr=[KA_PURE_DOPC[1]], fmt="o", ms=9, color="k",
                    ecolor="0.5", capsize=2, label="Pure DOPC")
    ax.set_xlabel(XLABEL, fontsize=10)
    ax.set_ylabel(YLABEL)
    ax.text(0.03, 0.96, "(a)", transform=ax.transAxes, fontsize=13,
            fontweight="bold", va="top")
    ax.legend(fontsize=8.5, ncol=2)
    ax.tick_params(direction="in", top=True, right=True)

    # ---- (b) one panel per composition ------------------------------------
    CELLS = [(0, 2), (0, 3), (0, 4), (1, 2), (1, 3)]
    XLAB  = {2, 3, 4}          # panels with nothing beneath them in the grid
    lines = []
    for k, comp in enumerate(COMPS):
        r, c = CELLS[k]
        a2 = fig.add_subplot(gs[r, c])
        xs, ys = [], []
        for l in LIPIDS:
            xs.append(E[l][k]); ys.append(KA[l][k])
            a2.plot(E[l][k], KA[l][k], MARKERS[l], ms=7, color=COLORS[l],
                    markeredgecolor="k", markeredgewidth=0.5)
        if PURE_DOPC_TOTAL is not None:
            xs.append(PURE_DOPC_TOTAL); ys.append(KA_PURE_DOPC[0])
            a2.plot(PURE_DOPC_TOTAL, KA_PURE_DOPC[0], "o", ms=7, color="k")

        f = linfit(xs, ys)
        diag = cluster_diagnostic(k)
        if f:
            s, i, R2 = f
            xf = np.linspace(min(xs), max(xs), 50)
            a2.plot(xf, s * xf + i, ls="--", lw=1.3, color=FITCOL)
            a2.text(0.5, 0.05, f"R$^2$ = {R2:.3f}", transform=a2.transAxes,
                    ha="center", fontsize=9,
                    bbox=dict(facecolor="white", edgecolor="none",
                              alpha=0.85, pad=1.5))
            lines.append((comp, R2, diag))
        a2.text(0.95, 0.92, f"{comp}%", transform=a2.transAxes, ha="right",
                va="top", fontsize=11, fontweight="bold")
        a2.tick_params(direction="in", top=True, right=True, labelsize=9)
        a2.ticklabel_format(useOffset=False)
        if c == 2:
            a2.set_ylabel(YLABEL, fontsize=10)
        if k in XLAB:
            a2.set_xlabel(XSHORT, fontsize=8.5)
        if k == 0:
            a2.text(-0.30, 1.22, "(b)", transform=a2.transAxes, fontsize=13,
                    fontweight="bold", va="top", ha="left")

    plt.savefig("KA_vs_Total.png", dpi=600, bbox_inches="tight")
    plt.savefig("KA_vs_Total.pdf", bbox_inches="tight")
    plt.close(fig)
    print("wrote KA_vs_Total.png / .pdf")

    # ---- fit report -------------------------------------------------------
    with open("KA_vs_Total_fits.txt", "w") as fh:
        fh.write("K_A vs TOTAL lipid-lipid interaction energy\n")
        fh.write("Least-squares fit per composition, 4 systems each.\n\n")
        fh.write(f"{'DOPC%':>6}{'R^2':>8}{'dKA(PA-PG)':>13}"
                 f"{'within-fam':>12}{'ratio':>8}{'dx(PA-PG)':>11}"
                 f"{'dx(chain)':>11}\n")
        fh.write("-" * 69 + "\n")
        for comp, R2, d in lines:
            if d is None:
                fh.write(f"{comp:>5}%{R2:>8.3f}\n")
                continue
            sep, spread, ratio, xsep, xin = d
            fh.write(f"{comp:>5}%{R2:>8.3f}{sep:>13.1f}{spread:>12.1f}"
                     f"{ratio:>8.2f}{xsep:>11.2f}{xin:>11.2f}\n")
        fh.write(
            "\ndKA(PA-PG)  difference in mean K_A between the PA and PG "
            "families (mN/m)\n"
            "within-fam  mean |K_A| spread between the two chain lengths "
            "inside a family\n"
            "ratio       dKA / within-fam.  Below 1 means within-family "
            "scatter exceeds the\n"
            "            difference between families, so the line fits "
            "noise.\n"
            "dx(PA-PG)   separation of the PA and PG clusters "
            "(x10^3 kcal/mol)\n"
            "dx(chain)   x separation between the two CHAIN LENGTHS inside a "
            "family.  In the\n"
            "            partial figure this is ~0.15; here it is ~1, but "
            "the extra spread\n"
            "            is Ewald self-energy tracking atom count, not "
            "chain-length physics.\n\n"
            "The totals carry the PME self term on every diagonal subunit "
            "pair, so they are\n"
            "less suitable for this correlation than the partial energies.  "
            "Prefer\n"
            "KA_vs_Partial for the manuscript and use this as a supporting "
            "comparison.\n")
    print("wrote KA_vs_Total_fits.txt")
    for comp, R2, d in lines:
        tag = "" if (d is None or d[2] >= 1.0) else "   <-- fit not supported"
        print(f"  {comp}%  R^2 = {R2:.3f}" +
              ("" if d is None else f"   dKA={d[0]:.1f}  spread={d[1]:.1f}  "
                                    f"ratio={d[2]:.2f}") + tag)


if __name__ == "__main__":
    main()

In [ ]:
"""
Decomposed PARTIAL (i-j) interaction energies in the mixed lipid bilayers.

    (a) secondary-secondary   (DOPG, DOPA, DLPG, DLPA with themselves)
    (b) DOPC-DOPC
    (c) DOPC-secondary
    (d) overall lipid-lipid   = (a) + (b) + (c)

    python fig_decomposed_partial.py  ->  Decomposed_Partial_Energy.{png,pdf}
                                          Decomposed_Partial_Energy_table.txt

Self-contained: the energies are embedded below, so this runs with nothing
but numpy and matplotlib.

DEFINITION.  Following Langmuir 2025, 41, 2600-2618, each lipid is split into
four subunits -- headgroup, phosphate, glycerol, acyl tail -- and the partial
(i-j) energy sums only pairs between DIFFERENT subunits.  Head-head,
phos-phos, glyc-glyc and tail-tail are excluded.  That gives 24 pairs per
system: 6 within DOPC, 6 within the secondary lipid, 12 across.

PA headgroup = H12, the single P-OH proton (CHARMM type HOL, q = +0.34).

WHY PARTIAL RATHER THAN TOTAL.  GROMACS assigns the PME Ewald self-energy,
-(1/4pi eps0)(beta/sqrt pi) * sum_i q_i^2, to the DIAGONAL group pairs.  The
partial definition excludes every i-i pair and is therefore free of it; the
totals are not.  Two consequences: the totals are inflated (for 100 DOPG
lipids the self term is worth about -132,000 kJ/mol), and because it scales
with atom count it makes the totals depend on chain length for a
non-physical reason.

CAVEAT for the methods section.  Coul-SR is only the PME real-space part; the
reciprocal term is not decomposable by energy group and is absent throughout.
Cross terms between BONDED subunits also carry the intramolecular exclusion
correction, -q_i q_j erf(beta r)/r, which for opposite charges at ~1 Angstrom
is large and positive.  These are not purely intermolecular energies.

PROVENANCE.  Re-summed from the subunit reruns (201 frames, 2 ns spacing,
110 energy data sets) and reconciled against independent whole-lipid reruns.
Where both were run on identical frames -- DOPA 80/20 and 87/13 -- the
subunit sums reproduce the whole-lipid values to 0.0003% or better on the
Coulomb terms and exactly on LJ and ion terms.
"""
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ systems
COMPS  = [65, 72, 80, 87, 95]                 # mol % DOPC
LIPIDS = ["DOPG", "DOPA", "DLPG", "DLPA"]
PAIRS  = [("PG", "DOPG", "DLPG"), ("PA", "DOPA", "DLPA")]  # long, short chain

# CAPTION: compositions are 65/72/80/87/95 mol % DOPC.  The published series
# was 65/75/85/95 -- do not copy that caption.

COLORS = {"DOPG": "#E8000B", "DOPA": "#1F4E79",
          "DLPG": "#4C9A2A", "DLPA": "#E0A500"}

K = 4184.0            # kJ/mol -> x10^3 kcal/mol

PANEL_KEYS  = ["a", "b", "c", "d"]
PANEL_TITLE = {"a": "secondary–secondary", "b": "DOPC–DOPC",
               "c": "DOPC–secondary",      "d": "overall lipid–lipid"}

ELABEL = r"Interaction energy ($\times$10$^3$ kcal/mol)"
XLABEL = "DOPC content (mol %)"

# -------------------------------------------- partial (i-j) energy, kJ/mol
# Order per composition: [ (a) sec-sec, (b) DOPC-DOPC, (c) DOPC-sec, (d) overall ]
PARTIAL_KJ = {
 "DOPG": {65: [-61985.8,  -96779.7, -11872.8, -170638.3],
          72: [-49316.3, -108726.3, -10606.2, -168648.7],
          80: [-35448.2, -122290.8,  -8635.2, -166374.3],
          87: [-23127.1, -134916.5,  -6017.6, -164061.2],
          95: [ -8463.7, -150419.4,  -2433.5, -161316.6]},
 "DOPA": {65: [-15600.8,  -96794.8, -13952.9, -126348.5],
          72: [-12089.1, -108707.3, -12413.0, -133209.4],
          80: [ -8585.2, -122456.1,  -9828.4, -140869.7],
          87: [ -5461.5, -134901.3,  -7002.4, -147365.2],
          95: [ -1967.8, -150533.0,  -2781.3, -155282.1]},
 "DLPG": {65: [-61888.9,  -97294.2, -12108.2, -171291.3],
          72: [-49404.9, -109353.2, -10291.5, -169049.6],
          80: [-35399.0, -122531.6,  -8674.9, -166605.5],
          87: [-23075.2, -135154.1,  -5994.0, -164223.3],
          95: [ -8473.3, -150508.7,  -2348.2, -161330.1]},
 "DLPA": {65: [-15578.6,  -97150.0, -14235.3, -126963.9],
          72: [-12213.8, -109249.5, -12285.2, -133748.4],
          80: [ -8531.3, -122618.2,  -9909.4, -141058.9],
          87: [ -5513.8, -135261.4,  -6789.5, -147564.7],
          95: [ -1983.0, -150610.8,  -2718.4, -155312.2]},
}


# ------------------------------------------------------------------ checks
def check_sums(tol=1.0):
    """(d) must equal (a)+(b)+(c).  Catches a mistyped number."""
    bad = []
    for l in LIPIDS:
        for c in COMPS:
            a, b, cc, d = PARTIAL_KJ[l][c]
            if abs((a + b + cc) - d) > tol:
                bad.append((l, c, a + b + cc, d))
    if bad:
        print("!! (a)+(b)+(c) does not equal (d):")
        for l, c, s, d in bad:
            print(f"     {l} {c}%  sum={s:.1f}  stored d={d:.1f}")
    else:
        print(f"internal check OK: (a)+(b)+(c) = (d) for all "
              f"{len(LIPIDS) * len(COMPS)} systems")
    return not bad


def panels():
    """{panel: {lipid: [value per composition]}} in x10^3 kcal/mol."""
    out = {k: {l: [] for l in LIPIDS} for k in PANEL_KEYS}
    for l in LIPIDS:
        for c in COMPS:
            for k, v in zip(PANEL_KEYS, PARTIAL_KJ[l][c]):
                out[k][l].append(v / K)
    return out


# ------------------------------------------------------------------ figure
def main():
    check_sums()
    D = panels()

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.4))
    x = np.arange(len(COMPS))
    n = len(LIPIDS)
    w = 0.8 / n

    for ax, key, tag in zip(axes.flat, PANEL_KEYS, "abcd"):
        for i, l in enumerate(LIPIDS):
            ax.bar(x + (i - (n - 1) / 2) * w, D[key][l], w,
                   label=l, color=COLORS[l], edgecolor="black", linewidth=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{c}%" for c in COMPS])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_ylabel(ELABEL, fontsize=11)
        ax.set_xlabel(XLABEL, fontsize=11)
        ax.text(-0.13, 1.06, f"({tag})", transform=ax.transAxes, fontsize=14,
                fontweight="bold", va="top", ha="left")
        ax.text(0.5, 1.03, PANEL_TITLE[key], transform=ax.transAxes,
                ha="center", fontsize=11)
        ax.tick_params(direction="in", top=True, right=True)
        # every bar is negative, so open a band above zero for the legend
        # rather than letting it land on the data
        lo = min(min(D[key][l]) for l in LIPIDS)
        ax.set_ylim(lo * 1.06, abs(lo) * 0.26)
        ax.legend(fontsize=9, ncol=2, frameon=True, loc="upper right")

    plt.tight_layout()
    plt.savefig("Decomposed_Partial_Energy.png", dpi=600, bbox_inches="tight")
    plt.savefig("Decomposed_Partial_Energy.pdf", bbox_inches="tight")
    plt.close(fig)
    print("wrote Decomposed_Partial_Energy.png / .pdf")

    write_table(D)


# ------------------------------------------------------------------ table
def write_table(D, name="Decomposed_Partial_Energy_table.txt"):
    L = []
    L.append("Decomposed PARTIAL (i-j) lipid-lipid interaction energies")
    L.append("Mixed DOPC / secondary-lipid bilayers, gamma = 0 mN/m, 201 "
             "frames over 400 ns.")
    L.append("")
    L.append("Sum of the 24 UNLIKE subunit pairs per system: 6 within DOPC,")
    L.append("6 within the secondary lipid, 12 across.  Like-subunit pairs")
    L.append("(head-head, phos-phos, glyc-glyc, tail-tail) are excluded.")
    L.append("")
    L.append("  (a) secondary-secondary   (b) DOPC-DOPC")
    L.append("  (c) DOPC-secondary        (d) overall lipid-lipid = a+b+c")
    L.append("")

    # ---- main table, both units ------------------------------------------
    L.append("=" * 78)
    L.append("Table 1.  Energies in x10^3 kcal/mol")
    L.append("=" * 78)
    hdr = (f"{'lipid':<8}{'DOPC%':>7}{'(a) sec-sec':>14}{'(b) PC-PC':>13}"
           f"{'(c) PC-sec':>13}{'(d) overall':>14}")
    L += [hdr, "-" * len(hdr)]
    for l in LIPIDS:
        for c in COMPS:
            v = [t / K for t in PARTIAL_KJ[l][c]]
            L.append(f"{l:<8}{c:>6}%" + "".join(
                f"{t:>{wd}.2f}" for t, wd in zip(v, (14, 13, 13, 14))))
        L.append("")

    L.append("=" * 78)
    L.append("Table 2.  Same values in kJ/mol, as GROMACS reports them")
    L.append("=" * 78)
    hdr2 = (f"{'lipid':<8}{'DOPC%':>7}{'(a) sec-sec':>14}{'(b) PC-PC':>13}"
            f"{'(c) PC-sec':>13}{'(d) overall':>14}")
    L += [hdr2, "-" * len(hdr2)]
    for l in LIPIDS:
        for c in COMPS:
            v = PARTIAL_KJ[l][c]
            L.append(f"{l:<8}{c:>6}%" + "".join(
                f"{t:>{wd}.1f}" for t, wd in zip(v, (14, 13, 13, 14))))
        L.append("")

    # ---- chain-length comparison -----------------------------------------
    L.append("=" * 78)
    L.append("Table 3.  Chain-length comparison, 18:1 (DO) vs 12:0 (DL)")
    L.append("          100 x (|long| - |short|) / |short|, per cent")
    L.append("=" * 78)
    hdr3 = f"{'family':<8}{'panel':>7}" + "".join(f"{c:>9}%" for c in COMPS)
    L += [hdr3, "-" * len(hdr3)]
    worst = 0.0
    for fam, lng, sht in PAIRS:
        for k in PANEL_KEYS:
            row = []
            for j in range(len(COMPS)):
                a, b = D[k][lng][j], D[k][sht][j]
                p = 100 * (abs(a) - abs(b)) / abs(b)
                worst = max(worst, abs(p))
                row.append(f"{p:>9.2f} ")
            L.append(f"{fam:<8}{'(' + k + ')':>7}" + "".join(row))
        L.append("")
    L.append(f"Largest deviation anywhere: {worst:.2f} per cent.")
    L.append("Panel (d) is the tightest; panel (c) the loosest, as expected")
    L.append("since (c) is the smallest term and the only one built purely")
    L.append("from cross-species pairs.")
    L.append("")
    L.append("The partial energies are therefore essentially independent of")
    L.append("acyl chain length.  The headgroup does the discriminating: at")
    L.append("65% DOPC panel (d) separates PG from PA by about 35%, falling")
    L.append("to about 4% at 95% DOPC as the two families converge.")
    L.append("")
    L.append("NOTE.  The corresponding TOTAL energies do show a 3-6%")
    L.append("chain-length difference, but that is the Ewald self-energy")
    L.append("tracking atom count (DOPG 131 atoms per lipid vs DLPG 99), not")
    L.append("chain-length physics.  See fig_ka_vs_total.py.")

    with open(name, "w") as fh:
        fh.write("\n".join(L) + "\n")
    print("wrote", name)


if __name__ == "__main__":
    main()

In [ ]:
"""
Decomposed TOTAL interaction energies in the mixed lipid bilayers.

    (a) secondary-secondary   (DOPG, DOPA, DLPG, DLPA with themselves)
    (b) DOPC-DOPC
    (c) DOPC-secondary
    (d) overall lipid-lipid   = (a) + (b) + (c)

    python fig_decomposed_total.py  ->  Decomposed_Total_Energy.{png,pdf}
                                        Decomposed_Total_Energy_table.txt

Self-contained: the energies are embedded below, so this runs with nothing
but numpy and matplotlib.

DEFINITION.  TOTAL sums EVERY atom pair between the two lipids, taken from
whole-lipid reruns with energygrps = DOPC <secondary> IONS.  Contrast
fig_decomposed_partial.py, which sums only the 24 unlike subunit pairs and
excludes all like-subunit (i-i) terms.

=============================================================================
TWO CAVEATS THAT APPLY TO THE TOTALS BUT NOT TO THE PARTIAL ENERGIES.

1. EWALD SELF-ENERGY.  GROMACS assigns the PME self term,
   -(1/4pi eps0)(beta/sqrt pi) * sum_i q_i^2, to the DIAGONAL group pairs,
   which the totals include.  It is large: for 100 DOPG lipids sum q_i^2 is
   roughly 650 e^2, worth about -132,000 kJ/mol, which is the right size to
   account for the -100,827 kJ/mol in Coul-SR:DOPG-DOPG.  This is why panel
   (a) below is NEGATIVE where the published Figure 7(a) is positive -- the
   difference needs explaining, not presenting as a match.

2. THE TOTALS DEPEND STRONGLY ON CHAIN LENGTH; THE PARTIAL ENERGIES DO NOT.
   Table 3 below: up to 27% on panel (c) and 4-8% on panel (d), against
   under 0.5% for the partial energies on the same systems.

   The cause is LENNARD-JONES CONTACT, not the self-energy.  Comparing DOPG
   with DLPG at 65% DOPC, the Coulomb terms differ by at most 4% while the
   LJ terms differ by 28% (DOPC-secondary) and 44% (secondary-secondary):
   18:1 tails carry 100 atoms per lipid against 68 for 12:0, so they present
   far more van der Waals surface.  Tail-tail alone accounts for about 75%
   of the cross-term LJ.

   The partial definition excludes every like-subunit pair, tail-tail
   included, which removes exactly that dependence.  That -- not the self
   term -- is why the partial energies are chain-length invariant.

   (The self-energy IS nearly chain-length independent, because charge sits
   in the headgroup and phosphate, which are identical for DO and DL.  It
   sets the magnitude and sign of the diagonal terms, not their chain-length
   behaviour.)

Also: Coul-SR is only the PME real-space part; the reciprocal term is not
decomposable by energy group and is absent throughout.

Lead with the partial figure.  Use this one as a supporting comparison.
=============================================================================

PROVENANCE.  Whole-lipid reruns, 201 frames at 2 ns spacing over 400 ns.
DOPA 80/20 and 87/13 were run on IDENTICAL frames to their subunit reruns;
the subunit sums reproduce these totals to 0.0003% or better on the Coulomb
terms and exactly on the LJ and ion terms.
"""
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------ systems
COMPS  = [65, 72, 80, 87, 95]                 # mol % DOPC
LIPIDS = ["DOPG", "DOPA", "DLPG", "DLPA"]
PAIRS  = [("PG", "DOPG", "DLPG"), ("PA", "DOPA", "DLPA")]  # long, short chain

# CAPTION: compositions are 65/72/80/87/95 mol % DOPC.  The published series
# was 65/75/85/95 -- do not copy that caption.

COLORS = {"DOPG": "#E8000B", "DOPA": "#1F4E79",
          "DLPG": "#4C9A2A", "DLPA": "#E0A500"}

K = 4184.0            # kJ/mol -> x10^3 kcal/mol

PANEL_KEYS  = ["a", "b", "c", "d"]
PANEL_TITLE = {"a": "secondary–secondary", "b": "DOPC–DOPC",
               "c": "DOPC–secondary",      "d": "overall lipid–lipid"}

ELABEL = r"Interaction energy ($\times$10$^3$ kcal/mol)"
XLABEL = "DOPC content (mol %)"

# ------------------------------------------------------ total energy, kJ/mol
# Whole-lipid rerun averages, in the order
#   Coul:PC-PC, LJ:PC-PC, Coul:PC-sec, LJ:PC-sec, Coul:sec-sec, LJ:sec-sec
# Kept as components rather than pre-summed so the Coulomb / Lennard-Jones
# split stays available for Table 2 and the arithmetic happens in code.
RAW = {
 "DOPG": {65: [45019.9, -28410,   -4958.89, -22421.4, -100827,  -9759.63],
          72: [49194.1, -34103.1, -4358.45, -19494.7, -80593.1, -7038.99],
          80: [53637.6, -40710.9, -3577.05, -15651.1, -58391.5, -4400.82],
          87: [57586.3, -47184.7, -2544.44, -11061.1, -38230.9, -2468.97],
          95: [62082.6, -55573.1, -1053.73, -4414.56, -14073.7, -743.869]},
 "DOPA": {65: [45226.9, -28471,   -4489.71, -22208,   -60499.8, -8959.79],
          72: [49283.5, -34290.5, -3936.44, -19034.3, -48454.4, -6550.99],
          80: [53635.5, -40931.7, -3223.57, -15144.8, -35173.1, -4091.37],
          87: [57664.6, -47286.6, -2338.58, -10854.6, -23044.3, -2228.09],
          95: [62164.4, -55607.9, -949.57,  -4345.19, -8506.74, -645.619]},
 "DLPG": {65: [44959,   -30674.9, -5172.21, -17496.1, -101736,  -6768.17],
          72: [49097.5, -36124.1, -4614.56, -15118.3, -81281.2, -4913.37],
          80: [53636,   -42293.5, -3829.08, -12126.1, -58921.2, -3016.11],
          87: [57518.3, -48497.5, -2701.89, -8316.2,  -38551.8, -1813.68],
          95: [62215.9, -56049.5, -1111.97, -3362.99, -14183.5, -525.855]},
 "DLPA": {65: [45022.4, -30890.8, -4728.19, -16994.1, -61348.6, -6198.91],
          72: [49244,   -36166.4, -4207,    -15000,   -49132.1, -4261.39],
          80: [53651.2, -42638,   -3380.03, -11417.7, -35641.6, -2824.09],
          87: [57566.5, -48648.4, -2436.71, -7977.37, -23372.4, -1617.07],
          95: [62132.6, -56151.8, -1008.6,  -3170.62, -8613.16, -487.34]},
}

# index into the six components: (Coulomb, Lennard-Jones) for each panel
COMPONENT = {"b": (0, 1), "c": (2, 3), "a": (4, 5)}


# ------------------------------------------------------------------ helpers
def parts(l, c):
    """{panel: (coul, lj, total)} in kJ/mol, plus 'd' as the overall sum."""
    v = RAW[l][c]
    out = {k: (v[i], v[j], v[i] + v[j]) for k, (i, j) in COMPONENT.items()}
    out["d"] = (v[0] + v[2] + v[4], v[1] + v[3] + v[5], sum(v))
    return out


def panels():
    """{panel: {lipid: [value per composition]}} in x10^3 kcal/mol."""
    out = {k: {l: [] for l in LIPIDS} for k in PANEL_KEYS}
    for l in LIPIDS:
        for c in COMPS:
            p = parts(l, c)
            for k in PANEL_KEYS:
                out[k][l].append(p[k][2] / K)
    return out


def check_sums(tol=1e-6):
    """(a)+(b)+(c) must equal (d).  True by construction; verify anyway."""
    bad = []
    for l in LIPIDS:
        for c in COMPS:
            p = parts(l, c)
            if abs(p["a"][2] + p["b"][2] + p["c"][2] - p["d"][2]) > tol:
                bad.append((l, c))
    print(f"internal check {'OK' if not bad else 'FAILED'}: "
          f"(a)+(b)+(c) = (d) for all {len(LIPIDS) * len(COMPS)} systems")
    return not bad


# ------------------------------------------------------------------ figure
def main():
    check_sums()
    D = panels()

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.4))
    x = np.arange(len(COMPS))
    n = len(LIPIDS)
    w = 0.8 / n

    for ax, key, tag in zip(axes.flat, PANEL_KEYS, "abcd"):
        for i, l in enumerate(LIPIDS):
            ax.bar(x + (i - (n - 1) / 2) * w, D[key][l], w,
                   label=l, color=COLORS[l], edgecolor="black", linewidth=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{c}%" for c in COMPS])
        ax.axhline(0, color="k", lw=0.8)
        ax.set_ylabel(ELABEL, fontsize=11)
        ax.set_xlabel(XLABEL, fontsize=11)
        ax.text(-0.13, 1.06, f"({tag})", transform=ax.transAxes, fontsize=14,
                fontweight="bold", va="top", ha="left")
        ax.text(0.5, 1.03, PANEL_TITLE[key], transform=ax.transAxes,
                ha="center", fontsize=11)
        ax.tick_params(direction="in", top=True, right=True)

        # open a clear band for the legend instead of letting it sit on data.
        # panel (b) is positive here, the rest negative, so handle both.
        vals = [v for l in LIPIDS for v in D[key][l]]
        lo, hi = min(vals), max(vals)
        span = hi - lo
        if lo >= 0:
            ax.set_ylim(0, hi + 0.30 * span)
            loc = "upper right"
        else:
            ax.set_ylim(lo - 0.06 * span, max(hi, 0) + 0.30 * span)
            loc = "upper right"
        ax.legend(fontsize=9, ncol=2, frameon=True, loc=loc)

    plt.tight_layout()
    plt.savefig("Decomposed_Total_Energy.png", dpi=600, bbox_inches="tight")
    plt.savefig("Decomposed_Total_Energy.pdf", bbox_inches="tight")
    plt.close(fig)
    print("wrote Decomposed_Total_Energy.png / .pdf")

    write_table(D)


# ------------------------------------------------------------------ table
def write_table(D, name="Decomposed_Total_Energy_table.txt"):
    L = []
    L.append("Decomposed TOTAL lipid-lipid interaction energies")
    L.append("Mixed DOPC / secondary-lipid bilayers, gamma = 0 mN/m, 201 "
             "frames over 400 ns.")
    L.append("")
    L.append("All atom pairs between the two lipids, from whole-lipid reruns")
    L.append("(energygrps = DOPC <secondary> IONS).  Unlike the partial")
    L.append("energies, these INCLUDE the like-subunit (i-i) terms.")
    L.append("")
    L.append("  (a) secondary-secondary   (b) DOPC-DOPC")
    L.append("  (c) DOPC-secondary        (d) overall lipid-lipid = a+b+c")
    L.append("")

    # ---- Table 1: totals, x10^3 kcal/mol ---------------------------------
    L.append("=" * 78)
    L.append("Table 1.  Energies in x10^3 kcal/mol")
    L.append("=" * 78)
    hdr = (f"{'lipid':<8}{'DOPC%':>7}{'(a) sec-sec':>14}{'(b) PC-PC':>13}"
           f"{'(c) PC-sec':>13}{'(d) overall':>14}")
    L += [hdr, "-" * len(hdr)]
    for l in LIPIDS:
        for c in COMPS:
            p = parts(l, c)
            L.append(f"{l:<8}{c:>6}%" + "".join(
                f"{p[k][2] / K:>{wd}.2f}"
                for k, wd in zip(PANEL_KEYS, (14, 13, 13, 14))))
        L.append("")

    # ---- Table 2: Coulomb / LJ split, kJ/mol -----------------------------
    L.append("=" * 78)
    L.append("Table 2.  Coulomb and Lennard-Jones components, kJ/mol")
    L.append("          (Coul-SR is the PME real-space part only)")
    L.append("=" * 78)
    hdr2 = (f"{'lipid':<8}{'DOPC%':>7}{'panel':>7}{'Coul-SR':>13}"
            f"{'LJ-SR':>13}{'total':>13}")
    L += [hdr2, "-" * len(hdr2)]
    for l in LIPIDS:
        for c in COMPS:
            p = parts(l, c)
            for k in PANEL_KEYS:
                cou, lj, tot = p[k]
                L.append(f"{l:<8}{c:>6}%{'(' + k + ')':>7}"
                         f"{cou:>13.1f}{lj:>13.1f}{tot:>13.1f}")
            L.append("")

    # ---- Table 3: chain-length comparison --------------------------------
    L.append("=" * 78)
    L.append("Table 3.  Chain-length comparison, 18:1 (DO) vs 12:0 (DL)")
    L.append("          100 x (|long| - |short|) / |short|, per cent")
    L.append("=" * 78)
    hdr3 = f"{'family':<8}{'panel':>7}" + "".join(f"{c:>9}%" for c in COMPS)
    L += [hdr3, "-" * len(hdr3)]
    worst = 0.0
    for fam, lng, sht in PAIRS:
        for k in PANEL_KEYS:
            row = []
            for j in range(len(COMPS)):
                a, b = D[k][lng][j], D[k][sht][j]
                pc = 100 * (abs(a) - abs(b)) / abs(b)
                worst = max(worst, abs(pc))
                row.append(f"{pc:>9.2f} ")
            L.append(f"{fam:<8}{'(' + k + ')':>7}" + "".join(row))
        L.append("")
    L.append(f"Largest deviation anywhere: {worst:.2f} per cent.")
    L.append("")
    L.append("This is a LENNARD-JONES effect, not an electrostatic one.")
    L.append("Comparing DOPG with DLPG at 65 per cent DOPC:")
    L.append("")
    L.append("    panel        Coul diff    LJ diff    total diff")
    L.append("    DOPC-DOPC        +0.1%      -7.4%       +16.3%")
    L.append("    DOPC-sec         -4.1%     +28.2%       +20.8%")
    L.append("    sec-sec          -0.9%     +44.2%        +1.9%")
    L.append("")
    L.append("18:1 tails carry 100 atoms per lipid against 68 for 12:0, so")
    L.append("they present far more van der Waals surface.  Tail-tail alone")
    L.append("is about 75 per cent of the cross-term LJ.  The partial")
    L.append("definition excludes every like-subunit pair, tail-tail")
    L.append("included, which removes exactly this dependence -- that is why")
    L.append("the partial energies agree to under 0.5 per cent on panel (d).")
    L.append("See Decomposed_Partial_Energy_table.txt, Table 3.")
    L.append("")
    L.append("Note the Ewald self-energy is NOT the cause: it sits on the")
    L.append("Coulomb diagonal, and the Coulomb terms differ by at most 4 per")
    L.append("cent between chain lengths, because charge is concentrated in")
    L.append("the headgroup and phosphate which are identical for DO and DL.")
    L.append("What the self-energy does explain is the MAGNITUDE and SIGN of")
    L.append("the diagonal terms -- why panel (a) here is large and negative")
    L.append("where the published Figure 7(a) is positive.")

    with open(name, "w") as fh:
        fh.write("\n".join(L) + "\n")
    print("wrote", name)


if __name__ == "__main__":
    main()

In [ ]:
"""
Interaction energy between NaCl ions and the head / phosphate groups of the
secondary lipids, across the DOPC mixtures.

    python fig_ion_lipid.py  ->  Ion_Lipid_Energy.{png,pdf}
                                 Ion_Lipid_Energy_table.txt

Self-contained: the energies are embedded below, so this runs with nothing
but numpy and matplotlib.  (Supersedes the older ion_lipid_energy.py, which
imported figstyle.)

BAR CONVENTION, following Langmuir 2025, 41, 2600-2618, Figure 1:
    SOLID            ions <-> lipid HEADGROUP
    DIAGONAL STRIPE  ions <-> PHOSPHATE group

=============================================================================
THE PHOSPHATIDIC ACID HEADGROUP.

DOPA and DLPA are in the same position as DSPA: the "headgroup" left after
the phosphate is a single atom, H12, the P-OH proton (CHARMM type HOL,
q = +0.34).  So the question of whether to draw a solid bar applies here too.

The published figure omits the solid bar for DSPA.  Our H12-ion term is
however neither zero nor small, and it is POSITIVE -- a bare acidic proton
repelling Na+:

      kJ/mol        65/35   72/28   80/20   87/13    95/5
      DOPA head    +519.1  +335.4  +201.7   +89.7   +21.7
      DLPA head    +559.1  +381.6  +193.3  +105.1   +20.4

At 65/35 that is +134 kcal/mol against a phosphate of -936 kcal/mol: about
14% of the phosphate magnitude, acting in the opposite direction.  Omitting
it would hide a real contribution and would make the PA totals look more
attractive than they are.

PA_SHOW_HEAD below controls this.
    True  (default)  draw the PA solid bar.  Honest; departs from the
                     published figure, so say so in the caption.
    False            omit it, reproducing the published convention exactly.
                     The bar is then dropped from the plot only -- the value
                     still appears in the table, flagged.
=============================================================================

A NOTE ON STACKING.  For PA the two segments have OPPOSITE SIGNS (head
repulsive, phosphate attractive).  Stacking them on one running bottom makes
the positive cap overlap the negative base, so the bar looks like it reaches
the phosphate value when the net is smaller.  Positives are therefore
accumulated upward from zero and negatives downward, and a short black dash
marks the net.  The bar's extent is the span of the contributions; the net is
the dash, and is tabulated.
"""
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------------------------------- switches
PA_SHOW_HEAD = True      # see the block above before changing
PER_LIPID    = False     # False: whole-system energies, as the published
                         # figure plots them.  True: divide by the number of
                         # secondary lipids, which is the fairer comparison
                         # across compositions and is what the cooperativity
                         # argument uses.

# ----------------------------------------------------------------- systems
COMPS  = [65, 72, 80, 87, 95]                 # mol % DOPC
ORDER  = ["DOPG", "DOPA", "DLPG", "DLPA"]
NSEC   = {65: 100, 72: 80, 80: 58, 87: 38, 95: 14}   # secondary lipids/system
LONE_H = ("DOPA", "DLPA")     # headgroup is the single P-OH proton H12

# CAPTION: compositions are 65/72/80/87/95 mol % DOPC.  The published series
# was 65/75/85/95 -- do not copy that caption.

COLORS = {"DOPG": "#E8000B", "DOPA": "#1F4E79",
          "DLPG": "#4C9A2A", "DLPA": "#E0A500"}

KJ2KCAL = 1.0 / 4.184

# ------------------------------------------------------- ion energies, kJ/mol
# Coul-SR + LJ-SR between the NaCl ions and each subunit of the SECONDARY
# lipid.  Glycerol and tail are carried for the table; only head and
# phosphate are plotted.
ION = {
 "DOPG": {65: {"head": -454.85, "phos": -2278.91, "glyc": -949.38, "tail": 20.76},
          72: {"head": -317.95, "phos": -1509.83, "glyc": -642.54, "tail": 14.65},
          80: {"head": -180.87, "phos":  -858.66, "glyc": -339.50, "tail":  7.94},
          87: {"head":  -86.60, "phos":  -417.31, "glyc": -176.28, "tail":  3.95},
          95: {"head":  -25.78, "phos":   -98.24, "glyc":  -46.41, "tail":  0.59}},
 "DLPG": {65: {"head": -518.83, "phos": -2353.00, "glyc": -990.72, "tail": 13.83},
          72: {"head": -333.86, "phos": -1611.13, "glyc": -694.65, "tail": 10.01},
          80: {"head": -182.84, "phos":  -886.04, "glyc": -374.10, "tail":  4.51},
          87: {"head":  -95.79, "phos":  -424.66, "glyc": -190.25, "tail":  2.36},
          95: {"head":  -22.99, "phos":  -109.48, "glyc":  -53.18, "tail":  0.23}},
 "DOPA": {65: {"head":  519.13, "phos": -3554.88, "glyc": -963.14, "tail": 23.34},
          72: {"head":  335.35, "phos": -2338.48, "glyc": -619.89, "tail": 14.72},
          80: {"head":  201.65, "phos": -1417.11, "glyc": -344.98, "tail":  8.60},
          87: {"head":   89.67, "phos":  -622.29, "glyc": -183.69, "tail":  4.23},
          95: {"head":   21.65, "phos":  -160.71, "glyc":  -38.00, "tail":  1.45}},
 "DLPA": {65: {"head":  559.14, "phos": -3915.09, "glyc": -1042.33, "tail": 14.20},
          72: {"head":  381.62, "phos": -2662.75, "glyc":  -730.88, "tail":  9.16},
          80: {"head":  193.31, "phos": -1370.61, "glyc":  -385.74, "tail":  5.94},
          87: {"head":  105.08, "phos":  -766.48, "glyc":  -223.64, "tail":  2.32},
          95: {"head":   20.41, "phos":  -153.23, "glyc":   -53.53, "tail":  0.21}},
}

SUBUNITS = ("head", "phos", "glyc", "tail")


def val(lip, c, sub, kcal=True):
    v = ION[lip][c][sub]
    if PER_LIPID:
        v /= NSEC[c]
    return v * KJ2KCAL if kcal else v


# ------------------------------------------------------------------ figure
def main():
    segs  = ["head", "phos"]                 # solid, then striped
    hatch = {"head": "", "phos": "///"}
    alpha = {"head": 1.0, "phos": 0.80}

    n, w = len(ORDER), 0.8 / len(ORDER)
    x = np.arange(len(COMPS))

    fig, ax = plt.subplots(figsize=(9.6, 5.4))
    for i, lip in enumerate(ORDER):
        off = (i - (n - 1) / 2) * w
        pos = np.zeros(len(COMPS))           # running top for positive parts
        neg = np.zeros(len(COMPS))           # running bottom for negative
        for seg in segs:
            if seg == "head" and lip in LONE_H and not PA_SHOW_HEAD:
                continue                     # published convention
            v = np.array([val(lip, c, seg) for c in COMPS])
            base = np.where(v >= 0, pos, neg)
            ax.bar(x + off, v, w * 0.92, bottom=base,
                   color=COLORS[lip], alpha=alpha[seg], hatch=hatch[seg],
                   edgecolor="k", linewidth=0.6,
                   label=lip if seg == "head" else None)
            pos = pos + np.where(v >= 0, v, 0.0)
            neg = neg + np.where(v < 0, v, 0.0)
        ax.plot(x + off, pos + neg, marker="_", ms=7, mew=1.6, ls="none",
                color="k", zorder=5)

    # Set limits explicitly.  The PA headgroup bars are positive and small
    # next to the phosphate term, and relying on autoscale had clipped them
    # at y = 0 in an earlier version of this figure.
    allv = [val(lip, c, s) for lip in ORDER for c in COMPS
            for s in segs
            if not (s == "head" and lip in LONE_H and not PA_SHOW_HEAD)]
    lo, hi = min(allv), max(max(allv), 0.0)
    pad = 0.06 * (hi - lo)
    ax.set_ylim(lo - pad, hi + pad)

    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}/{100 - c}" for c in COMPS])
    ax.set_xlabel("DOPC / secondary lipid (mol %)", fontsize=12)
    ax.set_ylabel("Interaction energy (kcal mol$^{-1}$)"
                  + (" per anionic lipid" if PER_LIPID else ""), fontsize=12)
    ax.tick_params(direction="in", top=True, right=True)

    # legend: lipid colours, plus a key for the two fills
    from matplotlib.patches import Patch
    keys = [Patch(facecolor=COLORS[l], edgecolor="k", label=l) for l in ORDER]
    keys += [Patch(facecolor="0.75", edgecolor="k", label="headgroup"),
             Patch(facecolor="0.75", edgecolor="k", hatch="///",
                   label="phosphate")]
    # above the axes, so it can never sit on a bar
    ax.legend(handles=keys, fontsize=8.5, ncol=6, frameon=False,
              loc="lower left", bbox_to_anchor=(0.0, 1.01))

    if PA_SHOW_HEAD:
        ax.text(0.99, 0.985,
                "PA headgroup = P–OH proton, repulsive (above zero)",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7.5, color="0.35")

    plt.tight_layout()
    plt.savefig("Ion_Lipid_Energy.png", dpi=600, bbox_inches="tight")
    plt.savefig("Ion_Lipid_Energy.pdf", bbox_inches="tight")
    plt.close(fig)
    print("wrote Ion_Lipid_Energy.png / .pdf")
    write_table()


# ------------------------------------------------------------------ table
def write_table(name="Ion_Lipid_Energy_table.txt"):
    L = []
    L.append("Interaction energy between NaCl ions and the secondary lipids")
    L.append("DOPC / secondary-lipid bilayers, gamma = 0 mN/m, 201 frames "
             "over 400 ns.")
    L.append("Coul-SR + LJ-SR, whole system unless stated.")
    L.append("")
    L.append("Figure convention: SOLID = headgroup, DIAGONAL STRIPE = "
             "phosphate.")
    L.append(f"PA_SHOW_HEAD = {PA_SHOW_HEAD}"
             + ("  (PA solid bar drawn)" if PA_SHOW_HEAD else
                "  (PA solid bar omitted from the PLOT; values below "
                "are unaffected)"))
    L.append("")
    L.append("The DOPA / DLPA headgroup is a single atom: H12, the P-OH")
    L.append("proton (CHARMM HOL, q = +0.34).  Its ion interaction is")
    L.append("POSITIVE because a bare acidic proton repels Na+.  It is not")
    L.append("negligible -- at 65/35 it is about 14% of the phosphate term")
    L.append("and opposes it.")
    L.append("")

    L.append("=" * 78)
    L.append("Table 1.  Whole-system energies, kcal/mol")
    L.append("=" * 78)
    hdr = (f"{'lipid':<7}{'DOPC%':>7}{'head':>10}{'phos':>11}{'head+phos':>12}"
           f"{'glyc':>10}{'tail':>8}{'all four':>11}")
    L += [hdr, "-" * len(hdr)]
    for lip in ORDER:
        for c in COMPS:
            v = {s: ION[lip][c][s] * KJ2KCAL for s in SUBUNITS}
            hp = v["head"] + v["phos"]
            L.append(f"{lip:<7}{c:>6}%{v['head']:>10.1f}{v['phos']:>11.1f}"
                     f"{hp:>12.1f}{v['glyc']:>10.1f}{v['tail']:>8.1f}"
                     f"{sum(v.values()):>11.1f}")
        L.append("")

    L.append("=" * 78)
    L.append("Table 2.  Per anionic lipid, kJ/mol")
    L.append("          (secondary lipids per system: "
             + ", ".join(f"{c}%:{NSEC[c]}" for c in COMPS) + ")")
    L.append("=" * 78)
    hdr2 = (f"{'lipid':<7}{'DOPC%':>7}{'nsec':>6}{'head':>9}{'phos':>10}"
            f"{'head+phos':>12}{'all four':>11}")
    L += [hdr2, "-" * len(hdr2)]
    for lip in ORDER:
        for c in COMPS:
            v = {s: ION[lip][c][s] / NSEC[c] for s in SUBUNITS}
            L.append(f"{lip:<7}{c:>6}%{NSEC[c]:>6}{v['head']:>9.2f}"
                     f"{v['phos']:>10.2f}{v['head'] + v['phos']:>12.2f}"
                     f"{sum(v.values()):>11.2f}")
        L.append("")
    L.append("Binding per lipid weakens by about 3x from 65/35 to 95/5 in all")
    L.append("four lipids, while counterions per anionic lipid RISE from 1.34")
    L.append("to 3.43 (one per anionic lipid plus a fixed 34 NaCl pairs).")
    L.append("So the weakening is not ion scarcity; it follows the anionic")
    L.append("surface charge density.")
    L.append("")

    L.append("=" * 78)
    L.append("Table 3.  Headgroup comparison, PA vs PG (phosphate term)")
    L.append("          100 x (|PA| - |PG|) / |PG|, per cent, averaged over")
    L.append("          the two chain lengths")
    L.append("=" * 78)
    L.append(f"{'DOPC%':>7}{'PG phos/lipid':>16}{'PA phos/lipid':>16}"
             f"{'PA vs PG':>11}")
    L.append("-" * 50)
    for c in COMPS:
        pg = np.mean([ION[l][c]["phos"] / NSEC[c] for l in ("DOPG", "DLPG")])
        pa = np.mean([ION[l][c]["phos"] / NSEC[c] for l in ("DOPA", "DLPA")])
        L.append(f"{c:>6}%{pg:>16.2f}{pa:>16.2f}"
                 f"{100 * (abs(pa) - abs(pg)) / abs(pg):>+10.0f}%")
    L.append("")
    L.append("PA phosphate outbinds PG at every composition, 20/20 systems,")
    L.append("with no composition dependence.")

    with open(name, "w") as fh:
        fh.write("\n".join(L) + "\n")
    print("wrote", name)


if __name__ == "__main__":
    main()

In [ ]:
# %% [markdown]
# # Area Compressibility Modulus (K_A) - grouped bar chart
#
# Grouped bars of K_A (mN/m) per composition for each DOPC/secondary-lipid
# system, with error bars, and PURE DOPC as a dashed benchmark line with a
# shaded band.  Legend for the systems on top.
#
# VALUES ARE HARDCODED, NOT READ FROM EXCEL.
# "New Paper_Updated II.xlsx" (sheet "New Systems") still carries the DOPA row
# in its original order, in which the 80/20, 87/13 and 95/5 entries are rotated
# by one position.  Reading from it reproduces the error.  The values below are
# the corrected ones, each reconfirmed by rerunning K_A = kT<A>/Var(A) on the
# corresponding box_DOPA_*.xvg trajectory:
#
#        composition     spreadsheet      corrected
#        80/20             228.9            261.8
#        87/13             261.8            247.1
#        95/5              247.1            228.9
#
# DOPG, DLPG and DLPA were unaffected.

# %%  Setup
import numpy as np
import matplotlib.pyplot as plt

# %%  Data
#
# K_A from area fluctuations, K_A = k_B T <A> / Var(A), on the total box area
# at T = 298.15 K, over the full 400 ns production trajectory.
#
# KA_SEM   standard error over five consecutive 80 ns blocks  (what is plotted)
# KA_SD    block-to-block standard deviation = SEM * sqrt(5)
#
# Set ERRBAR = "sd" to show the block scatter instead of the standard error.
# The SEM is the uncertainty on the mean and is the correct bar for comparing
# systems; the SD describes how much a single 80 ns window varies and is the
# more conservative thing to show if a referee asks for spread.

COMPS = [65, 72, 80, 87, 95]

KA = {
    "DOPG": [236.7, 223.3, 217.3, 217.0, 260.0],
    "DOPA": [246.4, 250.5, 261.8, 247.1, 228.9],   # corrected
    "DLPG": [216.2, 209.3, 274.1, 246.0, 287.3],
    "DLPA": [224.9, 269.7, 221.0, 264.8, 262.0],
}
KA_SEM = {
    "DOPG": [35.9, 12.1, 14.9, 13.5, 20.5],
    "DOPA": [13.0, 38.5, 12.1, 17.2, 14.4],        # corrected, moved with K_A
    "DLPG": [23.6, 14.4, 16.5, 12.4, 30.3],
    "DLPA": [20.7, 12.8, 16.7, 16.0, 14.7],
}
KA_SD = {
    "DOPG": [80.2, 27.0, 33.4, 30.2, 45.8],
    "DOPA": [29.1, 86.1, 27.1, 38.5, 32.1],
    "DLPG": [52.9, 32.3, 36.8, 27.7, 67.7],
    "DLPA": [46.3, 28.6, 37.4, 35.8, 32.8],
}

DOPC_BENCH = (245.81, 9.85)      # pure DOPC, prior study
ERRBAR = "sem"                   # "sem" or "sd"

DATA = {lip: {c: (KA[lip][j], (KA_SEM if ERRBAR == "sem" else KA_SD)[lip][j])
              for j, c in enumerate(COMPS)}
        for lip in KA}

print("Pure DOPC benchmark:", DOPC_BENCH)
for k in DATA:
    print(k, {c: DATA[k][c] for c in sorted(DATA[k])})

# %%  Style
LIPID_ORDER = ["DOPG", "DOPA", "DLPG", "DLPA"]
COLORS = {
    "DOPG": "#E8000B",   # red
    "DOPA": "#1F4E79",   # dark blue
    "DLPG": "#A9D18E",   # light green
    "DLPA": "#FFD966",   # gold/yellow
}
YMIN = 170          # baseline; top is set automatically to the tallest error bar

# %%  Plot
comps = sorted({c for s in DATA.values() for c in s})   # 65,72,80,87,95
lipids = [l for l in LIPID_ORDER if l in DATA]
n = len(lipids)
x = np.arange(len(comps))
width = 0.8 / n

fig, ax = plt.subplots(figsize=(7.2, 5.4))

# pure-DOPC benchmark band + dashed mean (behind bars)
m, sd = DOPC_BENCH
ax.axhspan(m - sd, m + sd, color="#CFE8F7", alpha=0.7, zorder=0)   # band
ax.axhline(m, color="k", ls="--", lw=1.2, zorder=1)               # mean

for i, lip in enumerate(lipids):
    xs = x + (i - (n - 1) / 2) * width
    ys = [DATA[lip].get(c, (np.nan, 0))[0] for c in comps]
    es = [DATA[lip].get(c, (np.nan, 0))[1] for c in comps]
    ax.bar(xs, ys, width, yerr=es, capsize=3, label=lip,
           color=COLORS.get(lip), edgecolor="black", linewidth=0.8,
           error_kw=dict(ecolor="0.25", lw=1.0), zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([f"{c}%" for c in comps])
# top of y-axis = top of the tallest error bar (value + error), + small margin
ytop = max(DATA[l][c][0] + DATA[l][c][1] for l in lipids for c in DATA[l])
ax.set_ylim(YMIN, ytop + 2)
ax.set_ylabel("Area Compressibility Modulus (" + r"$\mathit{mN/m}$)", fontsize=12)
ax.tick_params(direction="out")
ax.legend(ncol=n, loc="upper center", bbox_to_anchor=(0.5, 1.08),
          frameon=True, fontsize=10, handlelength=1.2, columnspacing=1.2)
plt.tight_layout()
plt.savefig("KA_barplot.png", dpi=600, bbox_inches="tight")
plt.savefig("KA_barplot.pdf", bbox_inches="tight")
plt.show()